# Prompt Engineering Assignment: Cafe Order Processing System

This notebook converts natural-language cafe orders into strict JSON and follows the assignment's example code structure.


## Setup

If needed, install the OpenAI package in your uv project:

```powershell
uv add openai ipykernel
```

Set the API key in PowerShell before running:

```powershell
$env:OPENAI_API_KEY="your-key-here"
```


In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json

# Load environment variables
load_dotenv(override=True)

# Initialize LLM client
# Option 1: OpenAI (requires OPENAI_API_KEY)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


## System Prompt


In [3]:
SYSTEM_PROMPT = 'You are a cafe order processing assistant.\n\nConvert the customer\'s natural-language cafe order into strict JSON only.\nDo not include markdown, explanations, or extra text.\n\nUse exactly this structure:\n{\n  "items": [\n    {\n      "name": "Item name",\n      "quantity": 1,\n      "size": "regular",\n      "modifiers": []\n    }\n  ],\n  "total_items": 1\n}\n\nRules:\n1. Extract every ordered food or drink item.\n2. Convert written quantities such as "one", "two", and "three" into integers.\n3. If quantity is not stated, use 1.\n4. Normalize item names into clear singular title case, for example "hamburgers" -> "Hamburger" and "iced lattes" -> "Iced Latte".\n5. Allowed size values are "small", "regular", and "large".\n6. If no size is stated, use "regular".\n7. Put special instructions such as "decaf", "no sugar", or "extra hot" in the "modifiers" array.\n8. If a modifier applies to only part of a quantity, split it into separate line items.\n9. If the customer orders a meal with a named drink, keep the meal as one item and put the drink in modifiers, for example ["with Coke"].\n10. total_items must equal the sum of all item quantities.\n11. Do not invent items, quantities, sizes, or modifiers that the customer did not mention.\n12. Output must be valid JSON that can be parsed by json.loads().'

print(SYSTEM_PROMPT)


You are a cafe order processing assistant.

Convert the customer's natural-language cafe order into strict JSON only.
Do not include markdown, explanations, or extra text.

Use exactly this structure:
{
  "items": [
    {
      "name": "Item name",
      "quantity": 1,
      "size": "regular",
      "modifiers": []
    }
  ],
  "total_items": 1
}

Rules:
1. Extract every ordered food or drink item.
2. Convert written quantities such as "one", "two", and "three" into integers.
3. If quantity is not stated, use 1.
4. Normalize item names into clear singular title case, for example "hamburgers" -> "Hamburger" and "iced lattes" -> "Iced Latte".
5. Allowed size values are "small", "regular", and "large".
6. If no size is stated, use "regular".
7. Put special instructions such as "decaf", "no sugar", or "extra hot" in the "modifiers" array.
8. If a modifier applies to only part of a quantity, split it into separate line items.
9. If the customer orders a meal with a named drink, keep the mea

## Order Processing Function

The function follows the assignment structure: system message, user message, low temperature, and `json.loads()` parsing.


In [4]:
def process_order(customer_input):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": customer_input}
        ],
        temperature=0.1
    )
    return json.loads(response.choices[0].message.content)


## Five Example Inputs


In [7]:
orders = ['2x Americano, 1 large fries and 3 hamburger', "I'll have a cappuccino and two croissants please", 'Can I get three iced lattes, make one of them decaf', 'One burger meal with coke, and an extra order of fries', 'Two small flat whites, one extra hot, plus a large muffin']

for i, order in enumerate(orders, start=1):
    print(f"{i}. {order}")


1. 2x Americano, 1 large fries and 3 hamburger
2. I'll have a cappuccino and two croissants please
3. Can I get three iced lattes, make one of them decaf
4. One burger meal with coke, and an extra order of fries
5. Two small flat whites, one extra hot, plus a large muffin


## Run the Five Tests


In [8]:
for order in orders:
    print(f"Input: {order}")
    try:
        output = process_order(order)
        print("Output:")
        print(json.dumps(output, indent=2))
    except json.JSONDecodeError as e:
        print("Invalid JSON returned by the model:", e)
    except Exception as e:
        print("Error:", e)
    print("-" * 60)


Input: 2x Americano, 1 large fries and 3 hamburger
Output:
{
  "items": [
    {
      "name": "Americano",
      "quantity": 2,
      "size": "regular",
      "modifiers": []
    },
    {
      "name": "Fries",
      "quantity": 1,
      "size": "large",
      "modifiers": []
    },
    {
      "name": "Hamburger",
      "quantity": 3,
      "size": "regular",
      "modifiers": []
    }
  ],
  "total_items": 6
}
------------------------------------------------------------
Input: I'll have a cappuccino and two croissants please
Output:
{
  "items": [
    {
      "name": "Cappuccino",
      "quantity": 1,
      "size": "regular",
      "modifiers": []
    },
    {
      "name": "Croissant",
      "quantity": 2,
      "size": "regular",
      "modifiers": []
    }
  ],
  "total_items": 3
}
------------------------------------------------------------
Input: Can I get three iced lattes, make one of them decaf
Output:
{
  "items": [
    {
      "name": "Iced Latte",
      "quantity": 2,
   

## Expected JSON Outputs


### Example 1

**Input**

`2x Americano, 1 large fries and 3 hamburger`

**Expected JSON**

```json
{
  "items": [
    {
      "name": "Americano",
      "quantity": 2,
      "size": "regular",
      "modifiers": []
    },
    {
      "name": "Fries",
      "quantity": 1,
      "size": "large",
      "modifiers": []
    },
    {
      "name": "Hamburger",
      "quantity": 3,
      "size": "regular",
      "modifiers": []
    }
  ],
  "total_items": 6
}
```


### Example 2

**Input**

`I'll have a cappuccino and two croissants please`

**Expected JSON**

```json
{
  "items": [
    {
      "name": "Cappuccino",
      "quantity": 1,
      "size": "regular",
      "modifiers": []
    },
    {
      "name": "Croissant",
      "quantity": 2,
      "size": "regular",
      "modifiers": []
    }
  ],
  "total_items": 3
}
```


### Example 3

**Input**

`Can I get three iced lattes, make one of them decaf`

**Expected JSON**

```json
{
  "items": [
    {
      "name": "Iced Latte",
      "quantity": 2,
      "size": "regular",
      "modifiers": []
    },
    {
      "name": "Iced Latte",
      "quantity": 1,
      "size": "regular",
      "modifiers": [
        "decaf"
      ]
    }
  ],
  "total_items": 3
}
```


### Example 4

**Input**

`One burger meal with coke, and an extra order of fries`

**Expected JSON**

```json
{
  "items": [
    {
      "name": "Burger Meal",
      "quantity": 1,
      "size": "regular",
      "modifiers": [
        "with Coke"
      ]
    },
    {
      "name": "Fries",
      "quantity": 1,
      "size": "regular",
      "modifiers": []
    }
  ],
  "total_items": 2
}
```


### Example 5

**Input**

`Two small flat whites, one extra hot, plus a large muffin`

**Expected JSON**

```json
{
  "items": [
    {
      "name": "Flat White",
      "quantity": 1,
      "size": "small",
      "modifiers": [
        "extra hot"
      ]
    },
    {
      "name": "Flat White",
      "quantity": 1,
      "size": "small",
      "modifiers": []
    },
    {
      "name": "Muffin",
      "quantity": 1,
      "size": "large",
      "modifiers": []
    }
  ],
  "total_items": 3
}
```


## Optional Validation


In [ ]:
def validate_order_json(data):
    assert isinstance(data, dict)
    assert "items" in data and "total_items" in data
    assert isinstance(data["items"], list)
    assert isinstance(data["total_items"], int)

    calculated_total = 0
    for item in data["items"]:
        assert isinstance(item["name"], str)
        assert isinstance(item["quantity"], int) and item["quantity"] > 0
        assert item["size"] in {"small", "regular", "large"}
        assert isinstance(item["modifiers"], list)
        calculated_total += item["quantity"]

    assert calculated_total == data["total_items"]
    return True


In [ ]:
sample = {
    "items": [
        {"name": "Americano", "quantity": 2, "size": "regular", "modifiers": []},
        {"name": "Fries", "quantity": 1, "size": "large", "modifiers": []},
        {"name": "Hamburger", "quantity": 3, "size": "regular", "modifiers": []}
    ],
    "total_items": 6
}
print(validate_order_json(sample))
